# 00 · Preparación IA — Diagnóstico del dataset PatagonIA

**Base:** `origin/main` (HEAD `5101254`). El parquet leído aquí es **md5-idéntico**
al versionado en `origin/main` (`6d9bfab…`). Los scripts `01`–`05` **no se modifican**.

Este notebook es **solo diagnóstico**: genera números y los deja a la vista.
**No filtra, no imputa, no elige escenario, no toma decisiones.**

---

### ⚠️ Nota de disponibilidad de datos (afecta a la TAREA A)

El `dropna` sobre variables ambientales **no está en `02_elevacion_clima.py`** sino en
**`05_join_final.py:58`**:

```python
df = df.dropna(subset=["elevacion", "temp_media", "precip_anual",
                       "viento_medio", "humedad_relativa"])
```

`02` solo **crea** esas variables (y de hecho *ya reporta* sus nulos con `.isna().sum()`
en su última celda, sin dropear). El drop efectivo ocurre en `05`.

Para "reconstruir el dataset **sin** ese `dropna`" en sentido estricto haría falta el
**intermedio pre-drop** `data/processed/_intermedios/04_fuego_completo.parquet` (o
re-ejecutar `01`–`05` desde el FIRMS crudo). Ambos están **gitignoreados y ausentes**
del repo/disco: lo único versionado es el dataset **final, ya post-`dropna`**.

Este notebook, por tanto:
1. Busca el intermedio pre-drop; si existe, reconstruye de verdad sin `dropna`.
2. Si no existe (caso actual), usa el dataset versionado y **expone** que sobre los
   datos disponibles el `dropna` fue un **no-op** (0 nulos, 0 filas perdidas).
   Evidencia adicional: el docstring de `02` habla de *"nuestros 1.980 hexágonos"*,
   el mismo conteo que el dataset final ⇒ el drop no eliminó filas en la corrida real.


In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import skew, kurtosis

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)

# Variables ambientales sobre las que 05_join_final.py aplica el dropna
ENV_DROPNA = ["elevacion", "temp_media", "precip_anual", "viento_medio", "humedad_relativa"]

# Orden de columnas exacto de 05_join_final.py (para replicar el pipeline salvo el dropna)
COLUMNAS = [
    "hex", "lat", "lon",
    "n_focos", "brillo_medio", "brillo_max", "frp_medio", "frp_max",
    "brillo_t31_medio", "pct_noche", "pct_verano", "pct_conf_alta",
    "n_anios_activo", "mes_pico",
    "elevacion", "temp_media", "precip_anual", "viento_medio", "humedad_relativa",
    "cobertura_veg",
    "dist_asentamiento_km", "dist_ruta_km",
]

# Raíz del repo: sube desde el cwd hasta encontrar data/processed (portable:
# funciona tanto si se lanza desde la raíz como desde notebooks/).
def raiz_repo():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "processed").exists():
            return base
    return Path.cwd()
BASE = raiz_repo()
print("Raíz del repo   :", BASE)

# --- Carga robusta: preferimos el intermedio PRE-drop; si no, el dataset final ---
INTERMEDIO = BASE / "data/processed/_intermedios/04_fuego_completo.parquet"
FINAL_PARQ = BASE / "data/processed/patagonia_dataset.parquet"
FINAL_CSV  = BASE / "data/processed/patagonia_dataset.csv"

if INTERMEDIO.exists():
    FUENTE = f"intermedio PRE-drop: {INTERMEDIO}"
    raw = pd.read_parquet(INTERMEDIO)
    df_sin_dropna = raw[COLUMNAS].copy()          # mismo pipeline de 05, SIN el dropna
    PRE_DROP_DISPONIBLE = True
elif FINAL_PARQ.exists():
    FUENTE = f"dataset FINAL ya post-dropna: {FINAL_PARQ}"
    df_sin_dropna = pd.read_parquet(FINAL_PARQ)
    PRE_DROP_DISPONIBLE = False
else:
    FUENTE = f"dataset FINAL (csv) ya post-dropna: {FINAL_CSV}"
    df_sin_dropna = pd.read_csv(FINAL_CSV)
    PRE_DROP_DISPONIBLE = False

print("Fuente de datos :", FUENTE)
print("¿Intermedio pre-drop disponible? :", PRE_DROP_DISPONIBLE)
print("Shape cargado   :", df_sin_dropna.shape)


Raíz del repo   : C:\Users\juanc\OneDrive\Desktop\PatagonIA\PatagonIA
Fuente de datos : dataset FINAL ya post-dropna: C:\Users\juanc\OneDrive\Desktop\PatagonIA\PatagonIA\data\processed\patagonia_dataset.parquet
¿Intermedio pre-drop disponible? : False
Shape cargado   : (1980, 22)


## TAREA A — Recuperar nulos (reconstrucción sin `dropna`)

Se reconstruye el dataset con **el resto del pipeline igual** (mismo orden de columnas
de `05_join_final.py`) pero **sin** la línea `dropna(subset=env)`. Se reporta shape,
filas que se habrían perdido con el `dropna`, y % de nulos por columna.
**No se imputa nada** — los nulos quedan expuestos.


In [2]:
# Dataset reconstruido SIN dropna
df = df_sin_dropna

print("=== TAREA A ===")
print(f"Shape SIN dropna : {df.shape[0]:,} filas x {df.shape[1]} columnas")

# Cuántas filas eliminaría el dropna de 05_join_final.py sobre las 5 variables ambientales
mask_drop = df[ENV_DROPNA].isna().any(axis=1)
n_drop = int(mask_drop.sum())
print(f"Filas que perdería el dropna(subset=env) : {n_drop}")
print(f"Shape que quedaría CON dropna            : {df.shape[0]-n_drop:,} x {df.shape[1]}")

if not PRE_DROP_DISPONIBLE:
    print("\n[!] Fuente = dataset ya post-dropna. Por tanto n_drop observable = "
          f"{n_drop} (el drop de la corrida original NO es recuperable desde aquí).")
    print("    Para una recuperación real haría falta 04_fuego_completo.parquet (ausente).")


=== TAREA A ===
Shape SIN dropna : 1,980 filas x 22 columnas
Filas que perdería el dropna(subset=env) : 0
Shape que quedaría CON dropna            : 1,980 x 22

[!] Fuente = dataset ya post-dropna. Por tanto n_drop observable = 0 (el drop de la corrida original NO es recuperable desde aquí).
    Para una recuperación real haría falta 04_fuego_completo.parquet (ausente).


In [3]:
# % de nulos por columna (todas las columnas), ordenado desc
nul = pd.DataFrame({
    "n_nulos": df.isna().sum(),
    "pct_nulos": (100 * df.isna().mean()).round(3),
})
nul_ordenado = nul.sort_values("pct_nulos", ascending=False)
print("=== % de nulos por columna (sin imputar) ===")
nul_ordenado


=== % de nulos por columna (sin imputar) ===


,n_nulos,pct_nulos
hex,0,0.0
lat,0,0.0
lon,0,0.0
n_focos,0,0.0
brillo_medio,0,0.0
brillo_max,0,0.0
frp_medio,0,0.0
frp_max,0,0.0
brillo_t31_medio,0,0.0
pct_noche,0,0.0


In [4]:
# Foco en las 5 variables ambientales del dropna
print("=== Nulos en las variables ambientales del dropna ===")
env_nulos = pd.DataFrame({
    "n_nulos": df[ENV_DROPNA].isna().sum(),
    "pct_nulos": (100 * df[ENV_DROPNA].isna().mean()).round(3),
})
env_nulos


=== Nulos en las variables ambientales del dropna ===


,n_nulos,pct_nulos
elevacion,0,0.0
temp_media,0,0.0
precip_anual,0,0.0
viento_medio,0,0.0
humedad_relativa,0,0.0


## TAREA B — Cuantificar el *flaring*

Zona candidata de *flaring* (venteo de gas, cuenca neuquina / Vaca Muerta) definida
por bounding box: **`lat > -39.5 & lon > -70`**.


In [5]:
# Máscara de la zona de flaring por bbox
zona = (df.lat > -39.5) & (df.lon > -70)
n_zona = int(zona.sum())
n_tot = len(df)

print("=== TAREA B ===")
print("[B1] Hexágonos en zona (lat>-39.5 & lon>-70):", n_zona, f"de {n_tot}",
      f"= {100*n_zona/n_tot:.2f}% del total")

sum_zona = df.loc[zona, "n_focos"].sum()
sum_tot = df["n_focos"].sum()
print(f"[B2] n_focos en zona: {int(sum_zona):,} de {int(sum_tot):,}",
      f"= {100*sum_zona/sum_tot:.2f}% de la suma total de n_focos")


=== TAREA B ===
[B1] Hexágonos en zona (lat>-39.5 & lon>-70): 561 de 1980 = 28.33% del total
[B2] n_focos en zona: 98,870 de 174,371 = 56.70% de la suma total de n_focos


In [6]:
# [B3] Perfil medio: zona de flaring vs resto
perfil_cols = ["pct_noche", "pct_conf_alta", "n_anios_activo", "n_focos"]
perfil = pd.DataFrame({
    "zona_flaring": df.loc[zona, perfil_cols].mean(),
    "resto":        df.loc[~zona, perfil_cols].mean(),
})
perfil["ratio_zona/resto"] = perfil["zona_flaring"] / perfil["resto"]
print("=== [B3] Perfil medio: zona flaring vs resto ===")
perfil.round(4)


=== [B3] Perfil medio: zona flaring vs resto ===


,zona_flaring,resto,ratio_zona/resto
pct_noche,0.3399,0.2329,1.4589
pct_conf_alta,0.0930,0.0670,1.3884
n_anios_activo,5.5811,3.1586,1.7670
n_focos,176.2389,53.2072,3.3123


In [7]:
# [B4] Distribución de n_focos CON y SIN la zona de flaring
def resumen_dist(s):
    return pd.Series({
        "n":        len(s),
        "media":    s.mean(),
        "mediana":  s.median(),
        "p90":      s.quantile(0.90),
        "max":      s.max(),
        "asimetria":skew(s),               # skewness
        "curtosis": kurtosis(s),           # curtosis en exceso (Fisher); normal=0
    })

dist = pd.DataFrame({
    "con_flaring": resumen_dist(df["n_focos"]),
    "sin_flaring": resumen_dist(df.loc[~zona, "n_focos"]),
})
print("=== [B4] Distribución de n_focos (curtosis = exceso de Fisher) ===")
dist.round(3)


=== [B4] Distribución de n_focos (curtosis = exceso de Fisher) ===


,con_flaring,sin_flaring
n,1980.000,1419.000
media,88.066,53.207
mediana,11.000,6.000
p90,257.200,166.200
max,5406.000,1461.000
asimetria,10.781,4.448
curtosis,183.635,27.767


## TAREA C — Impacto sobre el target binario

`riesgo_alto = n_focos > 150`. % de positivos en tres escenarios, lado a lado:
- **completo**
- **sin bbox de flaring** (`lat > -39.5 & lon > -70`)
- **sin firma de flaring** (`pct_noche > 0.7 & pct_conf_alta < 0.05`), sin usar coordenadas


In [8]:
# Target binario
target = df["n_focos"] > 150

# Máscaras de exclusión
excl_bbox  = zona                                              # zona flaring por coords
excl_firma = (df.pct_noche > 0.7) & (df.pct_conf_alta < 0.05)  # firma, sin coords

def resumen_escenario(mask_incluir):
    d = target[mask_incluir]
    return pd.Series({
        "n_hex":       int(mask_incluir.sum()),
        "n_positivos": int(d.sum()),
        "pct_positivos": round(100 * d.mean(), 3),
    })

escenarios = pd.DataFrame({
    "completo":           resumen_escenario(pd.Series(True, index=df.index)),
    "sin_bbox_flaring":   resumen_escenario(~excl_bbox),
    "sin_firma_flaring":  resumen_escenario(~excl_firma),
})
print("=== TAREA C · % positivos (riesgo_alto = n_focos > 150) ===")
print(f"Hexágonos excluidos por bbox : {int(excl_bbox.sum())}")
print(f"Hexágonos excluidos por firma: {int(excl_firma.sum())}")
escenarios


=== TAREA C · % positivos (riesgo_alto = n_focos > 150) ===
Hexágonos excluidos por bbox : 561
Hexágonos excluidos por firma: 248


,completo,sin_bbox_flaring,sin_firma_flaring
n_hex,1980.000,1419.000,1732.000
n_positivos,328.000,158.000,289.000
pct_positivos,16.566,11.135,16.686


## TAREA D — Verificación de leakage

Contraste de magnitudes de correlación de `n_focos` contra:
- **features ambientales candidatas** (terreno/clima/presión humana), y
- **features de fuego** (derivadas de los mismos focos FIRMS que originan `n_focos`).

> Nota: la consigna dice *"8 features ambientales"* pero lista **7** columnas numéricas
> (`elevacion, temp_media, precip_anual, viento_medio, humedad_relativa,
> dist_asentamiento_km, dist_ruta_km`). Se usan esas 7. La 8.ª ambiental del dataset,
> `cobertura_veg`, es **categórica** y no admite correlación de Pearson directa, por lo
> que se reporta aparte con una V de Cramér.


In [9]:
# Features ambientales candidatas (numéricas, las 7 listadas)
ENV_FEATURES = ["elevacion", "temp_media", "precip_anual", "viento_medio",
                "humedad_relativa", "dist_asentamiento_km", "dist_ruta_km"]

corr_env = df[["n_focos"] + ENV_FEATURES].corr()["n_focos"].drop("n_focos")
tabla_env = pd.DataFrame({
    "corr_con_n_focos": corr_env.round(3),
    "abs": corr_env.abs().round(3),
}).sort_values("abs", ascending=False)
print("=== [D] Correlación n_focos vs features AMBIENTALES ===")
tabla_env


=== [D] Correlación n_focos vs features AMBIENTALES ===


,corr_con_n_focos,abs
viento_medio,-0.207,0.207
temp_media,0.193,0.193
humedad_relativa,-0.133,0.133
elevacion,-0.129,0.129
dist_asentamiento_km,-0.109,0.109
precip_anual,0.029,0.029
dist_ruta_km,0.016,0.016


In [10]:
# Features de FUEGO (derivadas de los mismos focos -> candidatas a leakage del target)
FIRE_FEATURES = ["brillo_medio", "brillo_max", "brillo_t31_medio",
                 "frp_medio", "frp_max",
                 "pct_noche", "pct_verano", "pct_conf_alta", "n_anios_activo"]

corr_fire = df[["n_focos"] + FIRE_FEATURES].corr()["n_focos"].drop("n_focos")
tabla_fire = pd.DataFrame({
    "corr_con_n_focos": corr_fire.round(3),
    "abs": corr_fire.abs().round(3),
}).sort_values("abs", ascending=False)
print("=== [D] Correlación n_focos vs features de FUEGO ===")
tabla_fire


=== [D] Correlación n_focos vs features de FUEGO ===


,corr_con_n_focos,abs
frp_max,0.383,0.383
n_anios_activo,0.366,0.366
brillo_max,0.278,0.278
pct_noche,0.247,0.247
pct_verano,0.148,0.148
frp_medio,0.125,0.125
pct_conf_alta,0.110,0.110
brillo_medio,-0.058,0.058
brillo_t31_medio,0.006,0.006


In [11]:
# Contraste de magnitudes (|r|) ambiental vs fuego
contraste = pd.DataFrame({
    "ambientales": [tabla_env["abs"].max(), tabla_env["abs"].mean(), tabla_env["abs"].median()],
    "fuego":       [tabla_fire["abs"].max(), tabla_fire["abs"].mean(), tabla_fire["abs"].median()],
}, index=["|r| max", "|r| medio", "|r| mediana"]).round(3)
print("=== [D] Contraste de magnitudes de |correlación| con n_focos ===")
contraste


=== [D] Contraste de magnitudes de |correlación| con n_focos ===


,ambientales,fuego
|r| max,0.207,0.383
|r| medio,0.117,0.191
|r| mediana,0.129,0.148


In [12]:
# Extra: cobertura_veg (categórica) vs target binario -> V de Cramér (referencia, no Pearson)
from scipy.stats import chi2_contingency
def cramers_v(a, b):
    tab = pd.crosstab(a, b)
    chi2 = chi2_contingency(tab)[0]
    n = tab.to_numpy().sum()
    r, k = tab.shape
    return np.sqrt((chi2 / n) / (min(r, k) - 1))

target = df["n_focos"] > 150
print("V de Cramér cobertura_veg vs (n_focos>150):",
      round(cramers_v(df["cobertura_veg"], target), 3))
print("(categórica: referencia; no comparable 1:1 con Pearson pero útil de contexto)")


V de Cramér cobertura_veg vs (n_focos>150): 0.183
(categórica: referencia; no comparable 1:1 con Pearson pero útil de contexto)


---
### Resumen de lo que este notebook dejó (sin decidir nada)

- **A:** dataset reconstruido según el orden de columnas de `05` **sin** `dropna`; se
  reportan shape, filas que perdería el `dropna` y % de nulos por columna. Sobre los
  datos versionados (post-drop, únicos disponibles) el `dropna` es un **no-op**.
- **B:** tamaño, concentración de `n_focos`, perfil medio y forma de la distribución de
  la zona de *flaring* vs el resto.
- **C:** % de positivos de `riesgo_alto` en los tres escenarios, lado a lado.
- **D:** correlaciones de `n_focos` con features ambientales vs de fuego, y su contraste.

Ninguna celda filtra, imputa ni elige escenario: solo expone números.


---
# PASO 1 — Verificación del crudo FIRMS `775078` + comparación contra Minería

> **Contexto (2026-07-18).** El equipo bajó un FIRMS crudo **nuevo**
> (`fire_archive_SV-C2_775078.csv`, en `data/raw/firms/`). El pipeline `01`–`05`
> del repo apunta a `fire_archive_SV-C2_767267.csv`, que **no está versionado**.
> Estas secciones **no modifican los scripts**: replican su lógica sobre el crudo nuevo
> para (1) verificar el crudo y (2) reconstruir el dataset **sin** el `dropna` de
> `05_join_final.py:58`. **No se corrige ni se filtra nada** — solo se reporta.

Referencia de Minería (dataset `767267` versionado): **1.980 hexágonos**, **174.371 focos**.


In [ ]:
# [P1] Verificación del crudo FIRMS 775078
import h3

RAW = BASE / "data/raw/firms/fire_archive_SV-C2_775078.csv"
RES_H3 = 5
LAT_MIN, LAT_MAX, LON_MIN, LON_MAX = -56, -38, -76, -62

crudo = pd.read_csv(RAW)
print(f"[P1] Shape crudo : {crudo.shape[0]:,} filas x {crudo.shape[1]} columnas")
print(f"[P1] Columnas    : {list(crudo.columns)}")
print("\n[P1] Dtypes:")
print(crudo.dtypes.to_string())

fechas = pd.to_datetime(crudo["acq_date"])
print(f"\n[P1] Rango acq_date : {fechas.min().date()}  ->  {fechas.max().date()}")
print(f"     Años presentes : {sorted(fechas.dt.year.unique().tolist())}")

print("\n[P1] confidence (l/n/h):")
print(pd.DataFrame({
    "n": crudo.confidence.value_counts(),
    "pct": (100*crudo.confidence.value_counts(normalize=True)).round(2),
}).to_string())

print("\n[P1] daynight:")
print(pd.DataFrame({
    "n": crudo.daynight.value_counts(),
    "pct": (100*crudo.daynight.value_counts(normalize=True)).round(2),
}).to_string())

print("\n[P1] Nulos por columna:")
print(pd.DataFrame({"n_nulos": crudo.isna().sum(),
                    "pct": (100*crudo.isna().mean()).round(3)}).to_string())


In [ ]:
# [P1] Agregación 01 sobre el crudo nuevo -> comparación contra Minería (1980 / 174.371)
f = crudo[(crudo.latitude < LAT_MAX) & (crudo.latitude > LAT_MIN) &
          (crudo.longitude > LON_MIN) & (crudo.longitude < LON_MAX)].copy()
f["hex"] = [h3.latlng_to_cell(la, lo, RES_H3) for la, lo in zip(f.latitude, f.longitude)]
agg_new = f.groupby("hex").size().rename("n_new").reset_index()

n_hex_new   = len(agg_new)
n_focos_new = int(agg_new.n_new.sum())
MIN_HEX, MIN_FOCOS = 1980, 174371

print(f"[P1] Focos fuera del bbox descartados : {len(crudo) - len(f):,}")
print("\n=== Nuevo crudo 775078  vs  Minería 767267 ===")
comp = pd.DataFrame({
    "hexagonos_con_foco": [n_hex_new, MIN_HEX, n_hex_new - MIN_HEX],
    "suma_n_focos":       [n_focos_new, MIN_FOCOS, n_focos_new - MIN_FOCOS],
}, index=["nuevo_775078", "mineria_767267", "diferencia"])
print(comp.to_string())
print(f"\nΔ hexágonos : {n_hex_new-MIN_HEX:+,}  ({100*(n_hex_new-MIN_HEX)/MIN_HEX:+.2f}%)")
print(f"Δ n_focos   : {n_focos_new-MIN_FOCOS:+,}  ({100*(n_focos_new-MIN_FOCOS)/MIN_FOCOS:+.2f}%)")

# Descomposición: ¿la diferencia está en hexágonos nuevos o infla los compartidos?
fin = pd.read_csv(BASE / "data/processed/patagonia_dataset.csv")[["hex", "n_focos"]] \
        .rename(columns={"n_focos": "n_old"})
m = agg_new.merge(fin, on="hex", how="outer", indicator=True)
shared  = m[m._merge == "both"]
newonly = m[m._merge == "left_only"]
oldonly = m[m._merge == "right_only"]
print("\n=== Descomposición de la diferencia ===")
print(f"Hexágonos en común (nuevo ∩ final)      : {len(shared):,}")
print(f"  · con n_focos idéntico                : {int((shared.n_new==shared.n_old).sum()):,}")
print(f"  · con n_focos distinto (todos ↑)      : {int((shared.n_new!=shared.n_old).sum()):,}")
print(f"Solo en crudo nuevo (562 esperados)     : {len(newonly):,}")
print(f"Solo en dataset final (esperado 0)      : {len(oldonly):,}")
print(f"\nFocos extra en los 1.980 compartidos    : {int(shared.n_new.sum()-shared.n_old.sum()):+,}")
print(f"Focos aportados por los hex NUEVOS       : {int(newonly.n_new.sum()):+,}")
print("\n[P1] NOTA: el crudo nuevo es SUPERCONJUNTO estricto del de Minería.")
print("     La diferencia NO se corrige — se reporta (consigna Paso 1).")


---
# PASO 2 — Reconstrucción del dataset SIN `dropna` (crudo `775078`)

Se re-ejecutó el pipeline `01`→`05` sobre el crudo nuevo **sin** la línea
`dropna(subset=env)` de `05_join_final.py:58` (los scripts del repo **no se tocaron**;
la corrida vive en `scratch_paso2_runner.py`, réplica exacta salvo el RAW y el drop).
Intermedios en `data/processed/_intermedios/`. Salida: **`data/processed/patagonia_ia_con_nulos.csv`**.

Esta es la recuperación de nulos **real** que la TAREA A no pudo hacer antes
(faltaba `04_fuego_completo.parquet`, ahora regenerado). **No se imputa ni se filtra.**


In [ ]:
# [P2] Dataset reconstruido SIN dropna: shape con/sin drop y % de nulos
CON_NULOS = BASE / "data/processed/patagonia_ia_con_nulos.csv"
dfn = pd.read_csv(CON_NULOS)

print(f"[P2] Shape SIN dropna : {dfn.shape[0]:,} filas x {dfn.shape[1]} columnas")

mask_null = dfn[ENV_DROPNA].isna().any(axis=1)   # ENV_DROPNA def. en celda de setup
n_drop = int(mask_null.sum())
print(f"[P2] Filas con >=1 nulo en las 5 ambientales : {n_drop}")
print(f"[P2] Shape que quedaría CON el dropna original : {dfn.shape[0]-n_drop:,} x {dfn.shape[1]}")
print(f"[P2] => el dropna habría descartado {n_drop} filas "
      f"({100*n_drop/len(dfn):.2f}% del dataset).")

# Filas con al menos un nulo en CUALQUIER columna (no solo las 5 ambientales)
n_any = int(dfn.isna().any(axis=1).sum())
print(f"[P2] Filas con >=1 nulo en cualquier columna  : {n_any} ({100*n_any/len(dfn):.2f}%)")

print("\n=== [P2] % de nulos por columna (sin imputar, desc) ===")
nul_all = pd.DataFrame({
    "n_nulos": dfn.isna().sum(),
    "pct_nulos": (100*dfn.isna().mean()).round(3),
}).sort_values("pct_nulos", ascending=False)
print(nul_all[nul_all.n_nulos > 0].to_string() or "(sin nulos)")

print("\n=== [P2] Nulos en las 5 variables ambientales del dropna ===")
print(pd.DataFrame({
    "n_nulos": dfn[ENV_DROPNA].isna().sum(),
    "pct_nulos": (100*dfn[ENV_DROPNA].isna().mean()).round(3),
}).to_string())


In [ ]:
# [P2] Perfil geográfico de las filas con nulos ambientales: ¿dónde están?
nulos = dfn[mask_null]
plenos = dfn[~mask_null]
print(f"[P2] Filas con nulos ambientales: {len(nulos)}  |  plenas: {len(plenos)}")

if len(nulos):
    print("\n=== Ubicación de las filas con nulos ===")
    print(f"lat  : min {nulos.lat.min():.2f}  med {nulos.lat.median():.2f}  max {nulos.lat.max():.2f}")
    print(f"lon  : min {nulos.lon.min():.2f}  med {nulos.lon.median():.2f}  max {nulos.lon.max():.2f}")

    # Comparación de medias nulos vs plenos en variables que SÍ tienen dato
    comp_cols = ["lat", "lon", "elevacion", "n_focos",
                 "dist_asentamiento_km", "dist_ruta_km"]
    comp_cols = [c for c in comp_cols if c in dfn.columns]
    perfil = pd.DataFrame({
        "nulos":  nulos[comp_cols].mean(numeric_only=True),
        "plenos": plenos[comp_cols].mean(numeric_only=True),
    })
    print("\n=== Perfil medio: filas con nulos vs plenas ===")
    print(perfil.round(2).to_string())

    # Proxy de 'costero': distancia (en grados) al borde este (Atlántico, lon=-62)
    # y al borde sur/norte del bbox. lon alto (cercano a -62) = borde atlántico.
    print("\n=== ¿Costeras? (cercanía al borde atlántico lon=-62) ===")
    print(f"lon máx del dataset entero : {dfn.lon.max():.3f}  (borde bbox = -62)")
    print(f"lon de las filas con nulos : {sorted(nulos.lon.round(2).tolist())}")

    # ¿Comparten celda madre res-4? (el clima se muestrea a res 4: una madre nula
    # vuelve nulos a todos sus hijos)
    nulos_madre = [h3.cell_to_parent(hx, 4) for hx in nulos.hex]
    from collections import Counter
    cnt = Counter(nulos_madre)
    print(f"\n=== Celdas madre res-4 involucradas: {len(cnt)} (para {len(nulos)} hijos nulos) ===")
    for madre, c in cnt.most_common():
        la, lo = h3.cell_to_latlng(madre)
        print(f"  madre {madre}  ({la:.2f},{lo:.2f})  -> {c} hexágonos hijos nulos")

    print("\n=== Detalle de las filas con nulos ===")
    print(nulos[["hex", "lat", "lon", "elevacion"] + ENV_DROPNA].to_string(index=False))
else:
    print("[P2] No hay filas con nulos ambientales en esta corrida.")
